<a href="https://colab.research.google.com/github/iamdavid2/subasta/blob/main/Auditar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Cargar el DataFrame desde el archivo Parquet de Google Drive
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ruta_archivo = '/content/drive/MyDrive/subasta_ganadera_data/data/combined/lotes.parquet'
try:
    df = pd.read_parquet(ruta_archivo)
    print(f"¡Datos cargados correctamente! Total de filas: {len(df)}")
except Exception as e:
    print(f"Error al cargar desde la ruta, usando variables del kernel: {e}")

# 2. Mostrar la información y resumen de los datos disponibles
print("\n--- RESUMEN DEL DATAFRAME COMPLETO ---")
print(df.info())

print("\n--- RESUMEN DEL DATAFRAME FILTRADO (df_filtrado) ---")
if 'df_filtrado' in locals() or 'df_filtrado' in globals():
    print(f"Filas en df_filtrado: {len(df_filtrado)}")
    print(df_filtrado.describe())
else:
    print("df_filtrado no está definido en el kernel actualmente.")

# 3. Mostrar las primeras filas del df para verificar
df.head()

¡Datos cargados correctamente! Total de filas: 60571

--- RESUMEN DEL DATAFRAME COMPLETO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60571 entries, 0 to 60570
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   feria_no       60571 non-null  object        
 1   fecha_subasta  60571 non-null  datetime64[ns]
 2   ciudad         60571 non-null  object        
 3   martillo       51100 non-null  object        
 4   pagina         60571 non-null  int64         
 5   lote           60571 non-null  int64         
 6   sexo           60571 non-null  object        
 7   cantidad       60571 non-null  int64         
 8   peso_total     60571 non-null  int64         
 9   peso_promedio  60571 non-null  int64         
 10  procedencia    60571 non-null  object        
 11  entrada        60571 non-null  object        
 12  precio_base    60571 non-null  int64         
 13  precio_final   60571 non-nul

,feria_no,fecha_subasta,ciudad,martillo,pagina,lote,sexo,cantidad,peso_total,peso_promedio,procedencia,entrada,precio_base,precio_final,observaciones
0,1693,2024-01-09,YOPAL,NELIDA OMAIRA GOMEZ DUE?AS,1,1,VG,1,530,530,SAN LUIS DE PALENQ,10:57:01,5500,5850,
1,1693,2024-01-09,YOPAL,NELIDA OMAIRA GOMEZ DUE?AS,1,2,VG,1,438,438,YOPAL,10:32:17,5700,6750,
2,1693,2024-01-09,YOPAL,NELIDA OMAIRA GOMEZ DUE?AS,1,3,VG,2,880,440,YOPAL,10:55:10,5400,5850,
3,1693,2024-01-09,YOPAL,NELIDA OMAIRA GOMEZ DUE?AS,1,4,MG,1,634,634,YOPAL,10:58:43,7500,8300,
4,1693,2024-01-09,YOPAL,NELIDA OMAIRA GOMEZ DUE?AS,1,5,MG,1,590,590,YOPAL,11:03:47,7600,7650,


In [3]:
# Filtrar los datos de enero de 2025 donde la columna sexo sea 'VG'
condicion_fecha = (df['fecha_subasta'] >= '2025-01-01') & (df['fecha_subasta'] <= '2025-01-31')
condicion_sexo = df['sexo'] == 'VG'

df_filtrado = df[condicion_fecha & condicion_sexo].copy()

# Mostrar total de registros filtrados y una vista previa
print(f"Total de registros encontrados para enero de 2025 con sexo 'VG': {len(df_filtrado)}")
display(df_filtrado.head())

Total de registros encontrados para enero de 2025 con sexo 'VG': 352


,feria_no,fecha_subasta,ciudad,martillo,pagina,lote,sexo,cantidad,peso_total,peso_promedio,procedencia,entrada,precio_base,precio_final,observaciones
19503,1799,2025-01-07,YOPAL,JUAN ALBERTO ANGARITA,1,2,VG,1,550,550,YOPAL,10:21:33,6200,7050,
19504,1799,2025-01-07,YOPAL,JUAN ALBERTO ANGARITA,1,3,VG,1,494,494,YOPAL,10:22:58,5900,6700,
19506,1799,2025-01-07,YOPAL,JUAN ALBERTO ANGARITA,1,5,VG,2,838,419,OROCUE,10:37:39,6200,6650,1 NG - VG
19510,1799,2025-01-07,YOPAL,JUAN ALBERTO ANGARITA,1,9,VG,2,1008,504,YOPAL,10:49:39,5700,6550,2 CON BAJA CONDICION CORPORAL
19527,1799,2025-01-07,YOPAL,JUAN ALBERTO ANGARITA,1,26,VG,3,1654,551,OROCUE,10:30:07,5900,6650,


In [38]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# @title 📊 Panel Interactivo de Filtros de Subasta
sexo = "VG" # @param ["VG", "MG", "HE", "HO", "Todos"] {allow-input: true}
fecha_desde = "2024-01-01" # @param {type:"date"}
fecha_hasta = "2025-12-31" # @param {type:"date"}
excluir_outliers = True # @param {type:"boolean"}
procedencia = "" # @param {type:"string"}
mostrar_promedio_por_lote = False # @param {type:"boolean"}
mostrar_promedio_ponderado = True # @param {type:"boolean"}

def plot_grafico_ferias_grande(
    sexo=None,
    fecha_desde=None,
    fecha_hasta=None,
    excluir_outliers=False,
    procedencia=None
):
    global df
    datos = df.copy()

    # Asegurar tipo datetime
    datos['fecha_subasta'] = pd.to_datetime(datos['fecha_subasta'])

    # Filtro de Fechas
    if fecha_desde:
        datos = datos[datos['fecha_subasta'] >= pd.to_datetime(fecha_desde)]
    if fecha_hasta:
        datos = datos[datos['fecha_subasta'] <= pd.to_datetime(fecha_hasta)]

    # Filtro de Sexo
    if sexo and sexo != 'Todos':
        datos = datos[datos['sexo'] == sexo]

    # Filtro de Procedencia
    if procedencia:
        datos = datos[datos['procedencia'].astype(str).str.contains(procedencia, case=False, na=False)]

    # Excluir Outliers (IQR)
    if excluir_outliers and not datos.empty:
        q1 = datos['precio_final'].quantile(0.25)
        q3 = datos['precio_final'].quantile(0.75)
        iqr = q3 - q1
        datos = datos[(datos['precio_final'] >= q1 - 1.5 * iqr) & (datos['precio_final'] <= q3 + 1.5 * iqr)]

    if not datos.empty:
        datos['precio_por_animal'] = datos['precio_final'] * datos['peso_promedio']

        # Agregar columna inteligente para AUDITAR
        media_precio = datos['precio_final'].mean()
        desviacion_precio = datos['precio_final'].std()
        if pd.isna(desviacion_precio) or desviacion_precio == 0:
            desviacion_precio = 1
        datos['auditar'] = (abs(datos['precio_final'] - media_precio) / desviacion_precio > 1.2).map({True: '⚠️ Revisión Requerida', False: '✅ Conforme'})
    else:
        datos['precio_por_animal'] = []
        datos['auditar'] = []

    return datos

# Obtener los datos filtrados
datos_procesados = plot_grafico_ferias_grande(
    sexo=sexo,
    fecha_desde=fecha_desde,
    fecha_hasta=fecha_hasta,
    excluir_outliers=excluir_outliers,
    procedencia=procedencia
)

if not datos_procesados.empty:
    # Calcular el valor total comercial de cada lote para ponderar: Precio/Kg * Peso Total Lote
    datos_procesados['valor_total_lote'] = datos_procesados['precio_final'] * datos_procesados['peso_total']

    # Agrupar de forma limpia evitando DeprecationWarning
    df_agrupado = datos_procesados.groupby('fecha_subasta', as_index=False).agg(
        valor_total_dia=('valor_total_lote', 'sum'),
        peso_total_dia=('peso_total', 'sum'),
        precio_kilo_promedio=('precio_final', 'mean')
    )
    df_agrupado['precio_ponderado'] = df_agrupado['valor_total_dia'] / df_agrupado['peso_total_dia']

    # Calcular Métricas Globales de la selección actual
    suma_pesos_global = datos_procesados['peso_total'].sum()
    suma_valor_global = datos_procesados['valor_total_lote'].sum()
    precio_ponderado_global = suma_valor_global / suma_pesos_global if suma_pesos_global > 0 else 0
    precio_promedio_simple = datos_procesados['precio_final'].mean()

    # Obtener Top 5 Mejores Lotes (Mayor precio_final)
    top_mejores = datos_procesados.sort_values(by='precio_final', ascending=False).head(5)

    # Obtener Top 5 Peores Lotes (Menor precio_final)
    top_peores = datos_procesados.sort_values(by='precio_final', ascending=True).head(5)

    # Crear gráfico interactivo
    fig = go.Figure()

    # Trazo 1: Precio Promedio por Lote
    if mostrar_promedio_por_lote:
        fig.add_trace(go.Scatter(
            x=df_agrupado['fecha_subasta'],
            y=df_agrupado['precio_kilo_promedio'],
            name=f"Precio Promedio por Lote ({sexo}) ($/Kg)",
            mode="lines+markers",
            marker=dict(size=8, color="#1f77b4"),
            hovertemplate=f"<b>Fecha:</b> %{{x}}<br><b>Sexo:</b> {sexo}<br><b>Precio Promedio por Lote:</b> $%{{y:,.0f}} COP/Kg<br><extra>Promedio por Lote</extra>"
        ))

    # Trazo 2: Precio Promedio PONDERADO por Kilo
    if mostrar_promedio_ponderado:
        fig.add_trace(go.Scatter(
            x=df_agrupado['fecha_subasta'],
            y=df_agrupado['precio_ponderado'],
            name=f"Precio Promedio Ponderado ({sexo}) ($/Kg)",
            mode="lines+markers",
            marker=dict(size=8, color="#2ca02c"),
            hovertemplate=f"<b>Fecha:</b> %{{x}}<br><b>Sexo:</b> {sexo}<br><b>Precio Ponderado:</b> $%{{y:,.0f}} COP/Kg<br><extra>Ponderado por Peso</extra>"
        ))

    # Configurar diseño optimizado
    fig.update_layout(
        title=dict(
            text=f"Comparación de Precios en Subasta - Sexo: {sexo}",
            font=dict(size=16)
        ),
        xaxis=dict(title="Fecha de Subasta", type="date", tickformat="%Y-%m-%d"),
        yaxis=dict(
            title="Precio por Kilogramo ($/Kg)",
            tickformat=",.0f"
        ),
        template="plotly_white",
        hovermode="x unified",
        width=1100,
        height=600,
        margin=dict(l=60, r=60, t=100, b=50),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.04,
            xanchor="center",
            x=0.5
        )
    )

    fig.show()

    # Imprimir resumen de métricas clave
    print("\n" + "="*60)
    print("📈 RESUMEN DE MÉTRICAS CLAVE (VALORES SELECCIONADOS)")
    print("="*60)
    print(f"• Precio Promedio por Lote Global   : ${precio_promedio_simple:,.0f} COP/Kg")
    print(f"• Precio Promedio Ponderado Global : ${precio_ponderado_global:,.0f} COP/Kg")
    print(f"• Peso Total Comercializado       : {suma_pesos_global:,.0f} Kg")
    print(f"• Valor Total Comercializado      : ${suma_valor_global:,.0f} COP")
    print("="*60)

    print("\n🏆 TOP 5 MEJORES LOTES VENDIDOS (Mayor Precio)")
    print("-"*60)
    for idx, row in top_mejores.iterrows():
        print(f"Lote {row['lote']} | Precio: ${row['precio_final']:,.0f} COP/Kg | Procedencia: {row['procedencia']} | Obs: {row['observaciones']}")

    print("\n📉 TOP 5 PEORES LOTES VENDIDOS (Menor Precio)")
    print("-"*60)
    for idx, row in top_peores.iterrows():
        print(f"Lote {row['lote']} | Precio: ${row['precio_final']:,.0f} COP/Kg | Procedencia: {row['procedencia']} | Obs: {row['observaciones']}")
    print("="*60)

    # Mostrar la tabla detallada
    print(f"\n📝 TABLA DETALLADA DE LOTES SELECCIONADOS ({len(datos_procesados)} registros):")
    columnas_mostrar = ['auditar', 'fecha_subasta', 'lote', 'sexo', 'cantidad', 'peso_total', 'peso_promedio', 'precio_base', 'precio_final', 'precio_por_animal', 'procedencia', 'observaciones']
    display(datos_procesados[columnas_mostrar].sort_values(by='fecha_subasta'))
else:
    print("❌ No se encontraron registros con los filtros seleccionados.")


📈 RESUMEN DE MÉTRICAS CLAVE (VALORES SELECCIONADOS)
• Precio Promedio por Lote Global   : $6,624 COP/Kg
• Precio Promedio Ponderado Global : $6,610 COP/Kg
• Peso Total Comercializado       : 12,209,362 Kg
• Valor Total Comercializado      : $80,709,161,100 COP

🏆 TOP 5 MEJORES LOTES VENDIDOS (Mayor Precio)
------------------------------------------------------------
Lote 226 | Precio: $8,300 COP/Kg | Procedencia: AGUAZUL | Obs: 
Lote 180 | Precio: $8,300 COP/Kg | Procedencia: YOPAL | Obs: 3 NG / 1 VG 1 CON MALFORMACION
Lote 285 | Precio: $8,300 COP/Kg | Procedencia: PORE | Obs: 
Lote 201 | Precio: $8,300 COP/Kg | Procedencia: YOPAL | Obs: 
Lote 8 | Precio: $8,300 COP/Kg | Procedencia: YOPAL | Obs: 

📉 TOP 5 PEORES LOTES VENDIDOS (Menor Precio)
------------------------------------------------------------
Lote 155 | Precio: $4,950 COP/Kg | Procedencia: YOPAL | Obs: 
Lote 91 | Precio: $4,950 COP/Kg | Procedencia: YOPAL | Obs: 
Lote 300 | Precio: $5,000 COP/Kg | Procedencia: YOPAL | Obs: 

,auditar,fecha_subasta,lote,sexo,cantidad,peso_total,peso_promedio,precio_base,precio_final,precio_por_animal,procedencia,observaciones
0,⚠️ Revisión Requerida,2024-01-09,1,VG,1,530,530,5500,5850,3100500,SAN LUIS DE PALENQ,
95,✅ Conforme,2024-01-09,96,VG,3,1256,419,5500,6100,2555900,MANI,
96,✅ Conforme,2024-01-09,97,VG,3,1382,461,6000,6750,3111750,MANI,
98,⚠️ Revisión Requerida,2024-01-09,99,VG,3,1146,382,5500,5750,2196500,YOPAL,2VG Y 1VC
99,⚠️ Revisión Requerida,2024-01-09,100,VG,4,1448,362,5500,5850,2117700,YOPAL,
...,...,...,...,...,...,...,...,...,...,...,...,...
41830,✅ Conforme,2025-12-23,111,VG,4,1590,398,6300,6950,2766100,MANI,3VG - 1NG
41829,✅ Conforme,2025-12-23,110,VG,4,1606,402,6400,7000,2814000,MANI,
41825,⚠️ Revisión Requerida,2025-12-23,105,VG,2,872,436,6700,7650,3335400,YOPAL,1VG - 1NG
41900,✅ Conforme,2025-12-23,185,VG,3,1140,380,6300,7100,2698000,SAN LUIS DE PALENQ,


### 📊 Comparación Directa entre Dos Fechas de Subasta

In [24]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# @title 📅 Panel Interactivo: Diagnóstico de Dirección y Tendencia del Mercado
inicio_periodo_a = "2026-10-01" # @param {type:"date"}
fin_periodo_a = "2025-10-31" # @param {type:"date"}
inicio_periodo_b = "2026-10-01" # @param {type:"date"}
fin_periodo_b = "2026-10-31" # @param {type:"date"}
sexo_filtro = "Todos" # @param ["Todos", "VG", "MG", "HE", "HO", "VC"] {allow-input: true}

# Función para filtrar los datos dinámicamente según el panel
def filtrar_datos_tendencia(inicio, fin, sexo):
    global df
    temp_df = df[(df['fecha_subasta'] >= pd.to_datetime(inicio)) & (df['fecha_subasta'] <= pd.to_datetime(fin))].copy()
    if sexo != 'Todos':
        temp_df = temp_df[temp_df['sexo'] == sexo]
    return temp_df

# Recargar los datos con los parámetros del panel
data_rango_a = filtrar_datos_tendencia(inicio_periodo_a, fin_periodo_a, sexo_filtro)
data_rango_b = filtrar_datos_tendencia(inicio_periodo_b, fin_periodo_b, sexo_filtro)

# Agrupar datos por fecha para analizar la tendencia diaria de precios
df_diario_a = data_rango_a.groupby('fecha_subasta')['precio_final'].mean().reset_index().sort_values('fecha_subasta')
df_diario_b = data_rango_b.groupby('fecha_subasta')['precio_final'].mean().reset_index().sort_values('fecha_subasta')

# Crear un índice relativo (Día 1, Día 2, ...)
df_diario_a['Dia_Relativo'] = range(1, len(df_diario_a) + 1)
df_diario_b['Dia_Relativo'] = range(1, len(df_diario_b) + 1)

# 1. Calcular precios iniciales, finales reales y variaciones
precio_ini_a = df_diario_a['precio_final'].iloc[0] if not df_diario_a.empty else 0
precio_fin_a = df_diario_a['precio_final'].iloc[-1] if not df_diario_a.empty else 0
var_pct_a = ((precio_fin_a - precio_ini_a) / precio_ini_a * 100) if precio_ini_a > 0 else 0

precio_ini_b = df_diario_b['precio_final'].iloc[0] if not df_diario_b.empty else 0
precio_fin_b = df_diario_b['precio_final'].iloc[-1] if not df_diario_b.empty else 0
var_pct_b = ((precio_fin_b - precio_ini_b) / precio_ini_b * 100) if precio_ini_b > 0 else 0

# 2. Crear Gráfico con indicadores visuales
fig = go.Figure()

if not df_diario_a.empty:
    fig.add_trace(go.Scatter(
        x=df_diario_a['Dia_Relativo'],
        y=df_diario_a['precio_final'],
        mode='lines+markers+text',
        name=f'Período A ({var_pct_a:+.1f}%)',
        line=dict(color='#1f77b4', width=4),
        marker=dict(size=10),
        text=[f"${x:,.0f}" for x in df_diario_a['precio_final']],
        textposition="top center"
    ))

if not df_diario_b.empty:
    fig.add_trace(go.Scatter(
        x=df_diario_b['Dia_Relativo'],
        y=df_diario_b['precio_final'],
        mode='lines+markers+text',
        name=f'Período B ({var_pct_b:+.1f}%)',
        line=dict(color='#ff7f0e', width=4),
        marker=dict(size=10),
        text=[f"${x:,.0f}" for x in df_diario_b['precio_final']],
        textposition="bottom center"
    ))

fig.update_layout(
    title=f"<b>Diagnóstico de Dirección del Mercado: Período A vs Período B</b><br><sup>Muestra el cambio porcentual neto de inicio a fin (Filtro Sexo: {sexo_filtro})</sup>",
    xaxis_title="Días Transcurridos del Período",
    yaxis_title="Precio Promedio ($/Kg)",
    template="plotly_white",
    width=1100,
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

# 3. Consola de Diagnóstico Simplificado
print("\n" + "="*80)
print(f"📢 DIAGNÓSTICO DEL COMPORTAMIENTO ({sexo_filtro})")
print("="*80)
print(f"🟢 Período A ({inicio_periodo_a} al {fin_periodo_a}): Inició en ${precio_ini_a:,.1f} | Cerró en ${precio_fin_a:,.1f} | Var: {var_pct_a:+.2f}%")
print(f"🟢 Período B ({inicio_periodo_b} al {fin_periodo_b}): Inició en ${precio_ini_b:,.1f} | Cerró en ${precio_fin_b:,.1f} | Var: {var_pct_b:+.2f}%")
print("-"*80)

if var_pct_a > 0 and var_pct_b > 0:
    print(f"💡 Ambos mercados subieron. El Período {'B' if var_pct_b > var_pct_a else 'A'} tuvo un crecimiento relativo más fuerte.")
elif var_pct_a < 0 and var_pct_b < 0:
    print(f"⚠️ Ambos mercados bajaron. El Período {'B' if var_pct_b < var_pct_a else 'A'} sufrió la mayor desvalorización.")
else:
    print("🔀 Los mercados tomaron rumbos opuestos: uno subió mientras que el otro bajó.")
print("="*80)


📢 DIAGNÓSTICO DEL COMPORTAMIENTO (Todos)
🟢 Período A (2026-10-01 al 2025-10-31): Inició en $0.0 | Cerró en $0.0 | Var: +0.00%
🟢 Período B (2026-10-01 al 2026-10-31): Inició en $8,885.8 | Cerró en $9,019.0 | Var: +1.50%
--------------------------------------------------------------------------------
🔀 Los mercados tomaron rumbos opuestos: uno subió mientras que el otro bajó.
